In [ ]:
import os
import time
import torch
import numpy as np
import rasterio
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm
import joblib

from multikernel_model import AlbasUNet

# --- Config ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  
print(f"Using device: {device}")

# Paths adjusted for your setup
model_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/best_model_2.pth'
dir_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data'
output_dir = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/predictions'
# Load the saved scaler
scaler_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/minmax_scaler.joblib'
scaler = joblib.load(scaler_path)


# Features to use (Landsat bands)
features_sep = ["BLU", "GRN", "RED", "NIR", "SW1", "SW2"]
band_indices = [0, 1, 2, 3, 4, 5]

# Parameters
n_features = len(band_indices)
window_size = 8

# Load model
model = AlbasUNet()
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()

# Get list of BAP files and sort them by year
all_files = os.listdir(dir_path)
bap_files = [f for f in all_files if '_BAP.tif' in f]
bap_files.sort()  # Sort files by name (year)

print(f"Found {len(bap_files)} BAP files")
print("First few BAP files:", bap_files[:5])

if not bap_files:
    print("No BAP files found!")
    exit()

# Get metadata from first BAP file
sample_path = os.path.join(dir_path, bap_files[0])
with rasterio.open(sample_path) as src:
    height, width = src.height, src.width
    crs = src.crs
    transform = src.transform
    dtype = src.dtypes[0]
    #print(f"Raster dimensions: {height}x{width}")
    #print(f"Total pixels: {height * width}")

    # Initialize dummy array with correct dimensions for all bands
    dummy_array = np.full((height, width, n_features), -9999, dtype=dtype)

    # Clean up existing predictions if they exist
    import shutil
if os.path.exists(output_dir):
    print(f"Removing existing predictions from {output_dir}")
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

# Process each target year
years_to_process = list(range(1984 + window_size - 1, 2024))
print(f"Processing years: {years_to_process[0]} to {years_to_process[-1]}")

for target_year in tqdm(years_to_process, desc="Processing years"):
    print(f"\nProcessing year {target_year}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    output_pred_file = os.path.join(output_dir, 
                                  f"{target_year}_disturbed_undisturbed_pred_unet_lansatbands_w8.tif")
    
    # Remove the skip check to force reprocessing of all years

    # Create data cube with memory-efficient approach
    data_cube = np.zeros((height, width, n_features, window_size), dtype=np.float32)
    
    # Pre-open all files for the window to avoid repeated file operations
    open_files = {}
    for w in range(window_size):
        year = target_year - (window_size - 1 - w)
        bap_filename = f"{year}0801_LEVEL3_LNDLG_BAP.tif"
        bap_path = os.path.join(dir_path, bap_filename)
        
        if os.path.exists(bap_path):
            open_files[w] = rasterio.open(bap_path)
        else:
            print(f"Missing BAP file for year {year}, filling with nodata")
            data_cube[:, :, :, w] = dummy_array
    
    try:
        # Read all bands for each file at once
        for w, src in open_files.items():
            try:
                # Read all bands at once - properly handle band indices
                bands_data = src.read([i + 1 for i in band_indices])  # Add 1 to each index since rasterio is 1-indexed
                data_cube[:, :, :, w] = bands_data.transpose(1, 2, 0)  # Reshape to match our format
            except Exception as e:
                print(f"Error reading bands from year {target_year - (window_size - 1 - w)}: {e}")
                data_cube[:, :, :, w] = dummy_array
    finally:
        # Close all open files
        for src in open_files.values():
            src.close()

    # Clean invalid values
    data_cube = np.nan_to_num(data_cube, nan=-9999, posinf=-9999, neginf=-9999)

    # Reshape for model input
    pixels = height * width
    input_data = data_cube.reshape(pixels, n_features, window_size)
    #print(f"Input data shape after reshape: {input_data.shape}")

    # Create mask 
    valid_mask = (input_data != -9999)
    
    # Reshape to 2D for scaling
    input_data_2d = input_data.transpose(0, 2 ,1 ).reshape(-1, n_features)

    # scale only valid data
    valid_rows = ~np.all(input_data_2d == -9999, axis=1)
    input_data_2d[valid_rows] = scaler.transform(input_data_2d[valid_rows])

    #clip the scaled data 
    input_data_2d = np.clip(input_data_2d, 0, 1)

        
    # Reshape back to original format
    input_data = input_data_2d.reshape(pixels, window_size, n_features).transpose(0, 2, 1)

    # set invalid data to 0
    input_data[~valid_mask] = 0

    #convert to tensor
    input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)


    #print(f"Input tensor shape: {input_tensor.shape}")

    # Process in very large batches for maximum GPU utilization
    batch_size = 65536  # Using 2^16 for optimal GPU performance with 24GB VRAM
    predictions = []
    
    # Calculate total number of batches
    n_batches = (pixels + batch_size - 1) // batch_size
    
    # Process all predictions on GPU first
    predictions_gpu = []
    with torch.no_grad():
        for i in tqdm(range(0, pixels, batch_size), total=n_batches, desc="Processing batches", leave=False):
            batch = input_tensor[i:i+batch_size]
            output = model(batch)
            probs = torch.sigmoid(output)
            preds = (probs > 0.4)
            predictions_gpu.append(preds)
            
        # Concatenate all predictions on GPU
        predictions_gpu = torch.cat(predictions_gpu, dim=0)
        # Transfer to CPU all at once
        predictions = predictions_gpu.cpu().numpy()

    # Print shapes for debugging
    #print(f"Predictions shape before reshape: {predictions.shape}")

    # Make sure the total size matches
    total_pixels = predictions.shape[0]
    #print(f"Total pixels in predictions: {total_pixels}")
    #print(f"Required pixels: {height * width}")

    # Reshape with size check - now we'll have a separate prediction map for each timestep
    if total_pixels == height * width:
        # Reshape to [height, width, window_size]
        predictions_raster = predictions.reshape(height, width, window_size)
        # Convert to uint8 after reshaping
        predictions_raster = predictions_raster.astype(np.uint8)
    else:
        #print(f"Size mismatch: got {total_pixels} pixels, expected {height * width}")
        raise ValueError("Prediction size mismatch - cannot continue")

    # Convert to uint8 after reshaping
    predictions_raster = predictions_raster.astype(np.uint8)
    #print(f"Final predictions_raster shape: {predictions_raster.shape}")

    # Save predictions with robust settings
    meta = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': window_size,
        'dtype': rasterio.uint8,
        'crs': crs,
        'transform': transform,
        'compress': 'lzw',  # Changed back to lzw for better compatibility
        'tiled': True,
        'blockxsize': 256,
        'blockysize': 256,
        'interleave': 'band'  # Keep this simple
    }
    
    try:
        with rasterio.open(output_pred_file, 'w', **meta) as dst:
            # Write each timestep as a separate band
            for t in range(window_size):
                # Ensure data is valid uint8
                band_data = predictions_raster[:, :, t]
                if not np.isfinite(band_data).all():
                    print(f"Warning: Found non-finite values in band {t+1}, replacing with 0")
                    band_data = np.nan_to_num(band_data, nan=0, posinf=0, neginf=0)
                
                # Write band
                dst.write(band_data, t + 1)
                
                # Add band description
                year = target_year - (window_size - 1 - t)
                dst.set_band_description(t + 1, f"Predictions for year {year}")
                
        # Verify the file
        with rasterio.open(output_pred_file) as src:
            test_read = src.read(1, window=((0, 256), (0, 256)))
            print(f"\nVerified file writing - test block shape: {test_read.shape}")
            
    except Exception as e:
        print(f"\nError saving predictions: {e}")
        print("Trying alternative save method...")
        
        # Try alternative save method with minimal settings
        basic_meta = {
            'driver': 'GTiff',
            'height': height,
            'width': width,
            'count': window_size,
            'dtype': rasterio.uint8,
            'crs': crs,
            'transform': transform,
            'interleave': 'band'  # This is required
        }
        
        with rasterio.open(output_pred_file, 'w', **basic_meta) as dst:
            for t in range(window_size):
                dst.write(predictions_raster[:, :, t], t + 1)
                year = target_year - (window_size - 1 - t)
                dst.set_band_description(t + 1, f"Predictions for year {year}")

    print(f"Saved predictions for years {target_year-(window_size-1)} to {target_year} in {output_pred_file}")

Using device: cuda


/tmp/ipykernel_11306/2158383902.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Found 40 BAP files
First few BAP files: ['19840801_LEVEL3_LNDLG_BAP.tif', '19850801_LEVEL3_LNDLG_BAP.tif', '19860801_LEVEL3_LNDLG_BAP.tif', '19870801_LEVEL3_LNDLG_BAP.tif', '19880801_LEVEL3_LNDLG_BAP.tif']
Removing existing predictions from /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Multikernel_Model/predictions
Processing years: 1991 to 2023


Processing years:   0%|          | 0/33 [00:00<?, ?it/s]


Processing year 1991
